In [31]:


import urllib.request

url = "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv"
urllib.request.urlretrieve(url, "/workspaces/spark-test/spark_input/epd_snomed_202603.csv")


('/workspaces/spark-test/spark_input/epd_snomed_202603.csv',
 <http.client.HTTPMessage at 0x749447a78b90>)

In [32]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/workspaces/spark-test/spark_input/epd_snomed_202603.csv")

print("Row count:", df.count())

Row count: 18364409


In [33]:
df.printSchema()
df.show(5)

root
 |-- YEAR_MONTH: timestamp (nullable = true)
 |-- REGIONAL_OFFICE_NAME: string (nullable = true)
 |-- REGIONAL_OFFICE_CODE: string (nullable = true)
 |-- ICB_NAME: string (nullable = true)
 |-- ICB_CODE: string (nullable = true)
 |-- PCO_NAME: string (nullable = true)
 |-- PCO_CODE: string (nullable = true)
 |-- PRACTICE_NAME: string (nullable = true)
 |-- PRACTICE_CODE: string (nullable = true)
 |-- ADDRESS_1: string (nullable = true)
 |-- ADDRESS_2: string (nullable = true)
 |-- ADDRESS_3: string (nullable = true)
 |-- ADDRESS_4: string (nullable = true)
 |-- POSTCODE: string (nullable = true)
 |-- BNF_CHEMICAL_SUBSTANCE_CODE: string (nullable = true)
 |-- BNF_CHEMICAL_SUBSTANCE: string (nullable = true)
 |-- BNF_PRESENTATION_CODE: string (nullable = true)
 |-- BNF_PRESENTATION_NAME: string (nullable = true)
 |-- BNF_CHAPTER_PLUS_CODE: string (nullable = true)
 |-- QUANTITY: double (nullable = true)
 |-- ITEMS: integer (nullable = true)
 |-- TOTAL_QUANTITY: double (nullable = tr

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pathlib import Path

# ----------------------------------------------------
# 1. Spark session (Codespaces-optimised)
# ----------------------------------------------------
spark = SparkSession.builder \
    .appName("NHS_CSV_to_Parquet") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024) \
    .getOrCreate()

# ----------------------------------------------------
# 2. Full schema (still used for correct parsing)
# ----------------------------------------------------
schema = StructType([
    StructField("YEAR_MONTH", StringType(), True),
    StructField("REGIONAL_OFFICE_NAME", StringType(), True),
    StructField("REGIONAL_OFFICE_CODE", StringType(), True),
    StructField("ICB_NAME", StringType(), True),
    StructField("ICB_CODE", StringType(), True),
    StructField("PCO_NAME", StringType(), True),
    StructField("PCO_CODE", StringType(), True),
    StructField("PRACTICE_NAME", StringType(), True),
    StructField("PRACTICE_CODE", StringType(), True),
    StructField("ADDRESS_1", StringType(), True),
    StructField("ADDRESS_2", StringType(), True),
    StructField("ADDRESS_3", StringType(), True),
    StructField("ADDRESS_4", StringType(), True),
    StructField("POSTCODE", StringType(), True),
    StructField("BNF_CHEMICAL_SUBSTANCE_CODE", StringType(), True),
    StructField("BNF_CHEMICAL_SUBSTANCE", StringType(), True),
    StructField("BNF_PRESENTATION_CODE", StringType(), True),
    StructField("BNF_PRESENTATION_NAME", StringType(), True),
    StructField("BNF_CHAPTER_PLUS_CODE", StringType(), True),
    StructField("QUANTITY", DoubleType(), True),
    StructField("ITEMS", IntegerType(), True),
    StructField("TOTAL_QUANTITY", DoubleType(), True),
    StructField("ADQ_USAGE", DoubleType(), True),
    StructField("NIC", DoubleType(), True),
    StructField("ACTUAL_COST", DoubleType(), True),
    StructField("UNIDENTIFIED", StringType(), True),
    StructField("SNOMED_CODE", StringType(), True)
])

# ----------------------------------------------------
# 3. Keep ONLY required columns
# ----------------------------------------------------
KEEP_COLUMNS = [
    "YEAR_MONTH",
    "ICB_NAME",
    "ICB_CODE",
    "PRACTICE_NAME",
    "PRACTICE_CODE",
    "BNF_CHEMICAL_SUBSTANCE",
    "BNF_PRESENTATION_NAME",
    "BNF_CHAPTER_PLUS_CODE",
    "QUANTITY",
    "ITEMS",
    "ACTUAL_COST"
]

# ----------------------------------------------------
# 4. Paths
# ----------------------------------------------------
BASE_DIR = Path("/workspaces/spark-test")

input_path = BASE_DIR / "spark_input"
output_path = BASE_DIR / "spark_output" / "test"

csv_files = list(input_path.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found in spark_input folder")

input_file = str(csv_files[0])

print(f"Reading: {input_file}")
print(f"Writing to: {output_path}")

# ----------------------------------------------------
# 5. Read CSV + select required fields only
# ----------------------------------------------------
df = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv(input_file) \
    .select(*KEEP_COLUMNS)

# ----------------------------------------------------
# 6. Write Parquet
# ----------------------------------------------------
df.write \
    .mode("overwrite") \
    .parquet(str(output_path))

print("Done: CSV successfully converted to Parquet (filtered columns only)")

Reading: /workspaces/spark-test/spark_input/epd_snomed_202603.csv
Writing to: /workspaces/spark-test/spark_output/test


Done: CSV successfully converted to Parquet (filtered columns only)


In [35]:
df = spark.read.parquet("spark_output/test")


df.printSchema()
df.show(5)

root
 |-- YEAR_MONTH: timestamp (nullable = true)
 |-- ICB_NAME: string (nullable = true)
 |-- ICB_CODE: string (nullable = true)
 |-- PRACTICE_NAME: string (nullable = true)
 |-- PRACTICE_CODE: string (nullable = true)
 |-- BNF_CHEMICAL_SUBSTANCE: string (nullable = true)
 |-- BNF_PRESENTATION_NAME: string (nullable = true)
 |-- BNF_CHAPTER_PLUS_CODE: string (nullable = true)
 |-- QUANTITY: double (nullable = true)
 |-- ITEMS: integer (nullable = true)
 |-- ACTUAL_COST: double (nullable = true)

+-------------------+--------------------+--------+--------------------+-------------+----------------------+---------------------+---------------------+--------+-----+-----------+
|         YEAR_MONTH|            ICB_NAME|ICB_CODE|       PRACTICE_NAME|PRACTICE_CODE|BNF_CHEMICAL_SUBSTANCE|BNF_PRESENTATION_NAME|BNF_CHAPTER_PLUS_CODE|QUANTITY|ITEMS|ACTUAL_COST|
+-------------------+--------------------+--------+--------------------+-------------+----------------------+---------------------+-----

In [36]:
df = spark.read.parquet("/workspaces/spark-test/spark_output/test")

print("Row count:", df.count())

Row count: 18364409


In [ ]:
import urllib.request
import time
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# ----------------------------------------------------
# 1. Spark session
# ----------------------------------------------------
spark = SparkSession.builder \
    .appName("NHS_Auto_Ingestion") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024) \
    .getOrCreate()

# ----------------------------------------------------
# 2. Schema
# ----------------------------------------------------
schema = StructType([
    StructField("YEAR_MONTH", StringType(), True),
    StructField("REGIONAL_OFFICE_NAME", StringType(), True),
    StructField("REGIONAL_OFFICE_CODE", StringType(), True),
    StructField("ICB_NAME", StringType(), True),
    StructField("ICB_CODE", StringType(), True),
    StructField("PCO_NAME", StringType(), True),
    StructField("PCO_CODE", StringType(), True),
    StructField("PRACTICE_NAME", StringType(), True),
    StructField("PRACTICE_CODE", StringType(), True),
    StructField("ADDRESS_1", StringType(), True),
    StructField("ADDRESS_2", StringType(), True),
    StructField("ADDRESS_3", StringType(), True),
    StructField("ADDRESS_4", StringType(), True),
    StructField("POSTCODE", StringType(), True),
    StructField("BNF_CHEMICAL_SUBSTANCE_CODE", StringType(), True),
    StructField("BNF_CHEMICAL_SUBSTANCE", StringType(), True),
    StructField("BNF_PRESENTATION_CODE", StringType(), True),
    StructField("BNF_PRESENTATION_NAME", StringType(), True),
    StructField("BNF_CHAPTER_PLUS_CODE", StringType(), True),
    StructField("QUANTITY", DoubleType(), True),
    StructField("ITEMS", IntegerType(), True),
    StructField("TOTAL_QUANTITY", DoubleType(), True),
    StructField("ADQ_USAGE", DoubleType(), True),
    StructField("NIC", DoubleType(), True),
    StructField("ACTUAL_COST", DoubleType(), True),
    StructField("UNIDENTIFIED", StringType(), True),
    StructField("SNOMED_CODE", StringType(), True)
])


KEEP_COLUMNS = [
    "YEAR_MONTH",
    "ICB_NAME",
    "ICB_CODE",
    "PRACTICE_NAME",
    "PRACTICE_CODE",
    "BNF_CHEMICAL_SUBSTANCE",
    "BNF_PRESENTATION_NAME",
    "BNF_CHAPTER_PLUS_CODE",
    "QUANTITY",
    "ITEMS",
    "ACTUAL_COST"
]


# ----------------------------------------------------
# 3. Paths
# ----------------------------------------------------
BASE_DIR = Path("/workspaces/spark-test")

input_path = BASE_DIR / "spark_input"
output_path = BASE_DIR / "spark_output" / "all_data"

input_path.mkdir(parents=True, exist_ok=True)
output_path.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------
# 4. URLs
# ----------------------------------------------------
urls = [
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/58d59946-4257-4677-bbe5-1a38d45aca5a/download/epd_snomed_202602.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/55a20475-2b0a-4dd9-8adf-01558f97506c/download/epd_snomed_202601.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/543255f3-3355-4fb1-b1f2-27daab6af721/download/epd_snomed_202512.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/9ab2d5e6-76eb-4bbc-acc1-789952ae3454/download/epd_snomed_202511.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/d28455b2-7373-4d4a-8577-b28973ddba00/download/epd_snomed_202510.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/017e350f-c007-4d6f-8700-d8063e87a931/download/epd_snomed_202509.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/1200d488-d175-4fbd-827d-149ba65ea104/download/epd_snomed_202508.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/a63c5618-2af4-479c-9ae1-a3227d620ceb/download/epd_snomed_202507.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/943c3b10-3999-475a-b6b7-d77e1fcf8e8a/download/epd_snomed_202506.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/ede6bcf2-5d71-437f-a3fb-fc9817d7455c/download/epd_snomed_202505.csv",
    "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/a39bb2a2-189c-43ef-8783-2e77ccd794a0/download/epd_snomed_202504.csv"
]

# ----------------------------------------------------
# 5. PROCESS EACH FILE
# ----------------------------------------------------
first_file = True

for i, url in enumerate(urls, start=1):

    filename = url.split("/")[-1]
    file_path = input_path / filename

    print(f"\n========== FILE {i} OF {len(urls)} ==========")
    print(f"Downloading {filename}...")

    start = time.time()

    try:
        # Download
        urllib.request.urlretrieve(url, file_path)

        print("Reading into Spark...")

        df = spark.read \
        .option("header", True) \
        .option("mode", "PERMISSIVE") \
        .schema(schema) \
        .csv(str(file_path))

        df = df.select(*KEEP_COLUMNS)

        rows = df.count()
        print(f"Rows read: {rows:,}")

        print("Writing Parquet...")

        if first_file:
            df.write \
                .mode("overwrite") \
                .parquet(str(output_path))
            first_file = False
        else:
            df.write \
                .mode("append") \
                .parquet(str(output_path))

        # Delete CSV
        file_path.unlink()

        elapsed = (time.time() - start) / 60

        print(f"Completed in {elapsed:.1f} minutes.")
        print("CSV deleted.")

    except Exception as e:
        print(f"ERROR processing {filename}")
        print(e)

print("\n===================================")
print("Pipeline complete.")
print("All data written to:")
print(output_path)
print("===================================")

spark.stop()


========== FILE 1 OF 12 ==========
Reading into Spark...


Rows read: 18,364,409
Writing Parquet...


Completed in 6.1 minutes.
CSV deleted.

========== FILE 2 OF 12 ==========
Reading into Spark...


Rows read: 17,721,345
Writing Parquet...


Completed in 5.1 minutes.
CSV deleted.

========== FILE 3 OF 12 ==========
Reading into Spark...


...

Completed in 4.9 minutes.
CSV deleted.

========== FILE 11 OF 12 ==========
Reading into Spark...


Rows read: 18,287,310
Writing Parquet...


Completed in 4.8 minutes.
CSV deleted.

========== FILE 12 OF 12 ==========
Reading into Spark...


Rows read: 17,911,985
Writing Parquet...


Completed in 4.9 minutes.
CSV deleted.

Pipeline complete.
All data written to:
/workspaces/spark-test/spark_output/all_data


In [1]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NHS_Analysis") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.parquet("spark_output/all_data")

26/06/28 16:07:35 WARN Utils: Your hostname, codespaces-475009 resolves to a loopback address: 127.0.0.1; using 10.0.1.107 instead (on interface eth0)
26/06/28 16:07:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 16:07:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [41]:
df.groupBy("YEAR_MONTH") \
  .count() \
  .orderBy("YEAR_MONTH") \
  .show(50, False)

+-------------------+--------+
|YEAR_MONTH         |count   |
+-------------------+--------+
|2025-04-01 00:00:00|17911985|
|2025-05-01 00:00:00|18287310|
|2025-06-01 00:00:00|18133962|
|2025-07-01 00:00:00|18555470|
|2025-08-01 00:00:00|17796453|
|2025-09-01 00:00:00|18285462|
|2025-10-01 00:00:00|18453220|
|2025-11-01 00:00:00|17896076|
|2025-12-01 00:00:00|18452674|
|2026-01-01 00:00:00|18342436|
|2026-02-01 00:00:00|17721345|
|2026-03-01 00:00:00|18364409|
+-------------------+--------+



In [2]:
df.printSchema()
df.show(5)

root
 |-- YEAR_MONTH: timestamp (nullable = true)
 |-- ICB_NAME: string (nullable = true)
 |-- ICB_CODE: string (nullable = true)
 |-- PRACTICE_NAME: string (nullable = true)
 |-- PRACTICE_CODE: string (nullable = true)
 |-- BNF_CHEMICAL_SUBSTANCE: string (nullable = true)
 |-- BNF_PRESENTATION_NAME: string (nullable = true)
 |-- BNF_CHAPTER_PLUS_CODE: string (nullable = true)
 |-- QUANTITY: double (nullable = true)
 |-- ITEMS: integer (nullable = true)
 |-- ACTUAL_COST: double (nullable = true)



+-------------------+--------------------+--------+--------------+-------------+----------------------+---------------------+---------------------+--------+-----+-----------+
|         YEAR_MONTH|            ICB_NAME|ICB_CODE| PRACTICE_NAME|PRACTICE_CODE|BNF_CHEMICAL_SUBSTANCE|BNF_PRESENTATION_NAME|BNF_CHAPTER_PLUS_CODE|QUANTITY|ITEMS|ACTUAL_COST|
+-------------------+--------------------+--------+--------------+-------------+----------------------+---------------------+---------------------+--------+-----+-----------+
|2025-06-01 00:00:00|NHS SOUTH WEST LO...|     QWE|SUNRAY SURGERY|       H84618|  Duloxetine hydroc...| Duloxetine 30mg g...| 04: Central Nervo...|    56.0|    2|    4.37614|
|2025-06-01 00:00:00|NHS SOUTH WEST LO...|     QWE|SUNRAY SURGERY|       H84618|  Duloxetine hydroc...| Duloxetine 60mg g...| 04: Central Nervo...|    56.0|    2|     8.8248|
|2025-06-01 00:00:00|NHS SOUTH WEST LO...|     QWE|SUNRAY SURGERY|       H84618|  Duloxetine hydroc...| Duloxetine 60mg g...|

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load data
df = spark.read.parquet("spark_output/all_data")

# Clean + type safety
df = df.filter(F.col("BNF_CHAPTER_PLUS_CODE").isNotNull()) \
       .withColumn("ACTUAL_COST", F.col("ACTUAL_COST").cast("double"))

# National-level aggregation (practice is the unit)
practice_bnf = df.groupBy(
    "PRACTICE_CODE",
    "PRACTICE_NAME",
    "ICB_CODE",
    "ICB_NAME",
    "BNF_CHAPTER_PLUS_CODE"
).agg(
    F.sum("ACTUAL_COST").alias("total_cost"),
    F.sum("ITEMS").alias("total_items")
)

# Cost share within each practice (national comparability metric)
window_spec = Window.partitionBy("PRACTICE_CODE")

practice_bnf = practice_bnf.withColumn(
    "practice_total_cost",
    F.sum("total_cost").over(window_spec)
).withColumn(
    "cost_share",
    F.col("total_cost") / F.col("practice_total_cost")
)

# Optional performance optimisation
practice_bnf = practice_bnf.repartition(50)

# Write output
practice_bnf.write.mode("overwrite").parquet(
    "spark_output/practice_bnf_chapter"
)

In [5]:
practice_bnf.show(20, truncate=False)

+-------------+-----------------------------------+--------+-----------------------------------------------------------------------------+-------------------------------------------------------+------------------+-----------+-------------------+---------------------+
|PRACTICE_CODE|PRACTICE_NAME                      |ICB_CODE|ICB_NAME                                                                     |BNF_CHAPTER_PLUS_CODE                                  |total_cost        |total_items|practice_total_cost|cost_share           |
+-------------+-----------------------------------+--------+-----------------------------------------------------------------------------+-------------------------------------------------------+------------------+-----------+-------------------+---------------------+
|J83029       |TINKERS LANE SURGERY               |QOX     |NHS BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE BOARD|09: Nutrition and Blood                                |11

In [6]:
practice_bnf.groupBy("PRACTICE_CODE") \
      .agg(F.sum("cost_share").alias("sum_share")) \
      .show(10)

+-------------+------------------+
|PRACTICE_CODE|         sum_share|
+-------------+------------------+
|       Y08008|               1.0|
|       P85615|0.9999999999999998|
|       J81058|1.0000000000000002|
|       A84013|               1.0|
|       04C998|0.9999999999999996|
|       K82029|               1.0|
|       F81061|               1.0|
|       Y05704|               1.0|
|       Y00184|1.0000000000000004|
|       L84003|               1.0|
+-------------+------------------+
only showing top 10 rows

